## 01. Project Setup

Jalankan cell setup dari root proyek atau folder `notebooks`. Tidak ada training otomatis.


In [ ]:
from pathlib import Path
import sys
import warnings
import json
import os
import gc
import csv
from collections import Counter

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data/metadata/split_manifest.csv").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan; jalankan dari root atau notebooks.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from src.models.efficientnet_b2 import *
from src.data.dataloader import load_manifest_records, create_dataloader
from src.data.preprocessing import coerce_rgb_image, load_active_class_mapping
from src.callbacks import TrainingManager, EarlyStopping
from src.callbacks.lr_scheduler import build_scheduler
from src.callbacks.model_checkpoint import load_checkpoint
from src.training.resume import resume_from_checkpoint
from src.training.validate import validate_one_epoch
from src.evaluation.training_monitoring import load_training_history, history_review
from src.evaluation.visualization import plot_learning_curves
from src.inference.competition import audit_competition_test, competition_paths, load_submission_template
from src.inference.competition import CompetitionTestDataset, create_competition_dataloader
from src.inference.predict import predict_torch_classifier
from src.inference.submission import write_submission_csv, validate_submission_csv
print("PROJECT_ROOT:", PROJECT_ROOT)

## 02. Experiment Configuration

Semua flag eksekusi default `False`. Aktifkan hanya fase yang diinginkan, lalu jalankan ulang cell tersebut. Resume harus eksplisit; pilih batch 4 atau 2 jika VRAM tidak cukup.


In [ ]:
RUN_HEAD_TRAINING = False
RUN_FINE_TUNING = False
RUN_VALIDATION_EVALUATION = False
RUN_TEST_INFERENCE = False
RESUME_HEAD = False
RESUME_FINE_TUNING = False
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
NUM_WORKERS = 0
SEED = 42
CONFIG = load_experiment_config(PROJECT_ROOT)
set_global_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE), "| VRAM GiB:", round(torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3, 2))
else:
    warnings.warn("CUDA tidak tersedia: fallback CPU, training akan jauh lebih lambat.")
print("Device:", DEVICE, "| AMP:", DEVICE.type == "cuda", "| Config:", CONFIG)

## 03. Dataset Audit

Hanya manifest V1 aktif. Verifikasi path, isi train/validation dan hitung internal test tanpa melakukan evaluasi.


In [ ]:
manifest = pd.read_csv(PROJECT_ROOT / "data/metadata/split_manifest.csv", dtype=str)
split_counts = manifest.groupby("split").size().to_dict()
assert split_counts == {"train": 1340, "val": 383, "test_internal": 191}, split_counts
assert manifest["source_relative_path"].str.replace("\\", "/", regex=False).str.startswith("data/extracted/v1/train/").all()
assert not manifest["source_relative_path"].str.contains("/v5/", case=False, regex=False).any()
train_records = load_manifest_records("train", PROJECT_ROOT)
val_records = load_manifest_records("val", PROJECT_ROOT)
internal_records = load_manifest_records("test_internal", PROJECT_ROOT)
assert len(internal_records) == 191
for record in (*train_records, *val_records):
    with Image.open(PROJECT_ROOT / record.processed_relative_path) as image:
        image.verify()
CLASS_MAPPING = load_active_class_mapping(PROJECT_ROOT)
EXPECTED_CLASSES = {"cardboard", "electronic", "foodwaste", "glass", "metal", "paper", "plastic", "textile"}
assert set(CLASS_MAPPING) == EXPECTED_CLASSES and set(CLASS_MAPPING.values()) == set(range(8))
print("Split:", split_counts, "| Train/validation dapat dibaca:", len(train_records) + len(val_records))
print("Mapping aktif:", CLASS_MAPPING)

## 04. Class Distribution

Distribusi berasal dari manifest; split internal test hanya dihitung sebagai audit, tidak dipakai memilih model.


In [ ]:
distribution = pd.crosstab(manifest["class_name"], manifest["split"])
display(distribution)

## 05. EfficientNet-B2 Preprocessing

Validation dan inference memakai transform resmi bobot ImageNet: RGB, resize dan center crop 288, bicubic, float32 dan normalisasi ImageNet satu kali.


In [ ]:
TRAIN_TRANSFORM, VALIDATION_TRANSFORM, PREPROCESSING = build_efficientnet_b2_transforms()
print(PREPROCESSING)
print("Official transform:", VALIDATION_TRANSFORM)
print("Validasi synthetic:", validate_preprocessing(VALIDATION_TRANSFORM))

## 06. Training Augmentation Preview

Preview memakai satu gambar train; flip, rotasi <=15 derajat, shear <=10 derajat dan brightness <=10%. Tinjau apakah crop atau affine menghilangkan ciri kelas.


In [ ]:
example = coerce_rgb_image(PROJECT_ROOT / train_records[0].processed_relative_path)
figure, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(example); axes[0].set_title("Original train")
for index, axis in enumerate(axes[1:], start=1):
    tensor = TRAIN_TRANSFORM(example)
    restored = tensor * torch.tensor(IMAGENET_STD)[:, None, None] + torch.tensor(IMAGENET_MEAN)[:, None, None]
    axis.imshow(restored.clamp(0, 1).permute(1, 2, 0).numpy()); axis.set_title(f"Augmentasi {index}")
for axis in axes: axis.axis("off")
figure.tight_layout(); display(figure); plt.close(figure)

## 07. DataLoader Construction

Bangun loader hanya untuk train dan validation; `num_workers=0` aman untuk Windows/Jupyter.


In [ ]:
train_loader = create_dataloader("train", "efficientnet", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    seed=SEED, project_root=PROJECT_ROOT, model_config=CONFIG, transform=TRAIN_TRANSFORM, augmentation_enabled=False)
val_loader = create_dataloader("val", "efficientnet", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    seed=SEED, project_root=PROJECT_ROOT, model_config=CONFIG, transform=VALIDATION_TRANSFORM, augmentation_enabled=False)
print("Train:", len(train_loader.dataset), "Validation:", len(val_loader.dataset), "| Batch:", BATCH_SIZE)

## 08. Pretrained Model Initialization

Jalankan cell ini secara manual untuk membangun model. Bobot ImageNet akan diunduh oleh torchvision bila belum ada di Torch Hub cache; kegagalan unduh tidak diganti bobot acak.


In [ ]:
model = build_efficientnet_b2(8, pretrained=True)
freeze_for_head_training(model)
print("Pretrained terpasang:", WEIGHTS_NAME)
print("Torch Hub cache:", torch.hub.get_dir())

## 09. Model Summary

Ringkasan menjalankan satu forward sintetis, bukan training atau inference dataset.


In [ ]:
display(model_summary(model, batch_size=2, device=DEVICE))
print("Struktur classifier:", model.classifier)

## 10. Phase A Configuration

Backbone dan seluruh BatchNorm backbone dibekukan. Optimizer AdamW head, early stopping validation loss.


In [ ]:
HEAD_CONFIG = head_training_config()
HEAD_CONFIG.update({"batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS, "seed": SEED, "amp": DEVICE.type == "cuda"})
print(HEAD_CONFIG)

## 11. Phase A Manual Training

Set `RUN_HEAD_TRAINING=True` pada section 02 dan jalankan cell ini. Untuk melanjutkan run lama, set `RESUME_HEAD=True`; jangan jalankan lagi setelah early stop tanpa keputusan eksplisit.


In [ ]:
if RUN_HEAD_TRAINING:
    ensure_experiment_is_safe_to_start(PROJECT_ROOT, HEAD_EXPERIMENT_ID, resume=RESUME_HEAD)
    freeze_for_head_training(model)
    optimizer_a = build_head_optimizer(model)
    manager_a = TrainingManager(model=model, optimizer=optimizer_a, criterion=torch.nn.CrossEntropyLoss(),
        class_mapping=CLASS_MAPPING, model_identifier=MODEL_IDENTIFIER, experiment_id=HEAD_EXPERIMENT_ID,
        project_root=PROJECT_ROOT, device=DEVICE, training_config=HEAD_CONFIG, seed=SEED,
        early_stopping=EarlyStopping(patience=5, min_delta=0.0),
        scaler=torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda"),
        train_mode_callback=lambda active_model: apply_batch_norm_policy(active_model, trainable_batch_norm=False))
    start_a = 0
    if RESUME_HEAD:
        validate_resume_configuration(load_checkpoint(manager_a.checkpoints.last_path)["training_config"], HEAD_CONFIG)
        state = resume_from_checkpoint(manager_a.checkpoints.last_path, model, optimizer_a,
            early_stopping=manager_a.early_stopping, scaler=manager_a.scaler, model_identifier=MODEL_IDENTIFIER,
            class_mapping=CLASS_MAPPING, training_config=None, map_location="cpu")
        start_a, manager_a.history = state.start_epoch, list(state.history)
        manager_a.checkpoints.best_metric, manager_a.checkpoints.best_epoch = state.best_metric, state.best_epoch
        if manager_a.early_stopping.stopped: raise RuntimeError("Run Phase A sudah early-stop; tidak dilanjutkan otomatis.")
    try:
        manager_a.fit(train_loader, val_loader, 15, start_epoch=start_a, amp_enabled=True,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, gradient_clip_norm=1.0)
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("CUDA OOM. Hentikan, turunkan BATCH_SIZE menjadi 4 atau 2; gunakan gradient accumulation dan resume eksplisit. Training tidak dimulai ulang otomatis.") from error
    print("Best Phase A:", manager_a.checkpoints.best_path, "val_loss:", manager_a.checkpoints.best_metric)
    model.to("cpu"); del manager_a, optimizer_a
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Phase A tidak dijalankan.")

## 12. Phase A History & Learning Curves

Bisa dijalankan setelah restart kernel setelah section 01-02; tidak membuat grafik bila history belum ada.


In [ ]:
history_a_path = PROJECT_ROOT / "outputs/metrics" / f"{HEAD_EXPERIMENT_ID}.json"
if history_a_path.is_file():
    history_a = load_training_history(history_a_path)
    display(history_review(history_a)["all_epochs"])
    plot_learning_curves(history_a, PROJECT_ROOT / "outputs/figures" / f"{HEAD_EXPERIMENT_ID}_learning_curves.png", title="EfficientNet-B2 Phase A", display_figure=True)
else:
    print("History Phase A belum tersedia.")

## 13. Phase B Configuration

Muat best Phase A; buka hanya dua modul features terakhir. BatchNorm yang tetap frozen tidak mengubah running statistics.


In [ ]:
FINETUNE_CONFIG = fine_tuning_config()
FINETUNE_CONFIG.update({"batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS, "seed": SEED, "amp": DEVICE.type == "cuda"})
print(FINETUNE_CONFIG)

## 14. Phase B Manual Fine-Tuning

Set `RUN_FINE_TUNING=True` setelah Phase A. Optimizer dan scheduler baru; resume Phase B hanya jika `RESUME_FINE_TUNING=True`.


In [ ]:
if RUN_FINE_TUNING:
    ensure_experiment_is_safe_to_start(PROJECT_ROOT, FINETUNE_EXPERIMENT_ID, resume=RESUME_FINE_TUNING)
    if "model" in globals():
        model.to("cpu")
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    head_path = PROJECT_ROOT / "models/checkpoints" / HEAD_EXPERIMENT_ID / "best.pt"
    import hashlib
    FINETUNE_CONFIG["parent_head_sha256"] = hashlib.sha256(head_path.read_bytes()).hexdigest()
    loaded_head = load_efficientnet_b2_checkpoint(project_root=PROJECT_ROOT, checkpoint_path=head_path, device=DEVICE)
    model_b = loaded_head.model
    unfrozen = configure_partial_fine_tuning(model_b)
    optimizer_b = build_fine_tuning_optimizer(model_b)
    scheduler_b, scheduler_kind_b = build_scheduler(optimizer_b, {"name": "reduce_on_plateau", "mode": "min", "factor": 0.5, "patience": 4, "min_lr": 1e-7})
    manager_b = TrainingManager(model=model_b, optimizer=optimizer_b, criterion=torch.nn.CrossEntropyLoss(),
        class_mapping=CLASS_MAPPING, model_identifier=MODEL_IDENTIFIER, experiment_id=FINETUNE_EXPERIMENT_ID,
        project_root=PROJECT_ROOT, device=DEVICE, training_config=FINETUNE_CONFIG, seed=SEED,
        scheduler=scheduler_b, scheduler_kind=scheduler_kind_b, early_stopping=EarlyStopping(patience=10, min_delta=0.0),
        scaler=torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda"),
        train_mode_callback=lambda active_model: apply_batch_norm_policy(active_model, trainable_batch_norm=True))
    start_b = 0
    if RESUME_FINE_TUNING:
        validate_resume_configuration(load_checkpoint(manager_b.checkpoints.last_path)["training_config"], FINETUNE_CONFIG)
        state = resume_from_checkpoint(manager_b.checkpoints.last_path, model_b, optimizer_b, scheduler=scheduler_b,
            early_stopping=manager_b.early_stopping, scaler=manager_b.scaler, model_identifier=MODEL_IDENTIFIER,
            class_mapping=CLASS_MAPPING, training_config=None, map_location="cpu")
        start_b, manager_b.history = state.start_epoch, list(state.history)
        manager_b.checkpoints.best_metric, manager_b.checkpoints.best_epoch = state.best_metric, state.best_epoch
        if manager_b.early_stopping.stopped: raise RuntimeError("Run Phase B sudah early-stop; tidak dilanjutkan otomatis.")
    print("Blok trainable:", unfrozen, "| LR:", [group["lr"] for group in optimizer_b.param_groups])
    try:
        manager_b.fit(train_loader, val_loader, 45, start_epoch=start_b, amp_enabled=True,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, gradient_clip_norm=1.0)
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("CUDA OOM. Hentikan, turunkan BATCH_SIZE menjadi 4 atau 2; gunakan gradient accumulation dan resume eksplisit. Training tidak dimulai ulang otomatis.") from error
    print("Best Phase B:", manager_b.checkpoints.best_path, "val_loss:", manager_b.checkpoints.best_metric)
    model_b.to("cpu"); del manager_b, optimizer_b, scheduler_b
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Phase B tidak dijalankan.")

## 15. Phase A vs Phase B Comparison

Bandingkan best validation loss dari history aktual; Phase B tidak otomatis menang. Grafik dibuat hanya jika history tersedia.


In [ ]:
phase_rows = []
for experiment_id in (HEAD_EXPERIMENT_ID, FINETUNE_EXPERIMENT_ID):
    history_path = PROJECT_ROOT / "outputs/metrics" / f"{experiment_id}.json"
    if history_path.is_file():
        history = load_training_history(history_path)
        best = min(history, key=lambda row: float(row["val_loss"]))
        phase_rows.append({"experiment_id": experiment_id, "best_epoch": best["epoch"] + 1,
            "val_loss": best["val_loss"], "val_accuracy": best["val_accuracy"], "val_macro_f1": best["val_macro_f1"]})
        if experiment_id == FINETUNE_EXPERIMENT_ID:
            plot_learning_curves(history, PROJECT_ROOT / "outputs/figures" / f"{experiment_id}_learning_curves.png", title="EfficientNet-B2 Phase B", display_figure=True)
display(pd.DataFrame(phase_rows) if phase_rows else "Belum ada history training.")

## 16. Final Best Checkpoint Selection

Memilih minimum validation loss dari `best.pt` Phase A/B; metadata pointer disimpan tanpa menyalin bobot. Phase A saja sudah cukup.


In [ ]:
try:
    selected = select_best_phase_checkpoint(PROJECT_ROOT)
    selection_path = save_selected_checkpoint(PROJECT_ROOT, selected)
    print("Terpilih:", selected.phase, selected.checkpoint_path, "val_loss:", selected.val_loss, "| pointer:", selection_path)
except FileNotFoundError as error:
    print(error)

## 17. Validation Evaluation

Set `RUN_VALIDATION_EVALUATION=True` untuk evaluasi 383 validation dari checkpoint terpilih. Internal test tidak digunakan.


In [ ]:
evaluation = None
if RUN_VALIDATION_EVALUATION:
    selected = select_best_phase_checkpoint(PROJECT_ROOT)
    save_selected_checkpoint(PROJECT_ROOT, selected)
    evaluated_model = load_efficientnet_b2_checkpoint(project_root=PROJECT_ROOT, checkpoint_path=selected.checkpoint_path, device=DEVICE)
    evaluation = validate_one_epoch(evaluated_model.model, val_loader, torch.nn.CrossEntropyLoss(), CLASS_MAPPING, DEVICE)
    assert evaluation["sample_count"] == 383
    evaluation.update({"model_identifier": MODEL_IDENTIFIER, "phase": selected.phase, "checkpoint": str(selected.checkpoint_path.relative_to(PROJECT_ROOT)), "best_checkpoint_val_loss": selected.val_loss})
    metrics_path = PROJECT_ROOT / "outputs/metrics/efficientnet_b2_validation.json"
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
    display({key: evaluation[key] for key in ("loss", "accuracy", "macro_precision", "macro_recall", "macro_f1", "sample_count", "phase")})
    evaluated_model.model.to("cpu"); del evaluated_model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Evaluasi validation tidak dijalankan.")

## 18. Classification Report

Laporan semua kelas dari evaluasi validation yang sebenarnya.


In [ ]:
if evaluation is not None:
    display(pd.DataFrame(evaluation["classification_report"]).transpose())
else:
    print("Jalankan evaluasi validation dahulu.")

## 19. Confusion Matrix

Tampilkan confusion matrix mentah dan normalisasi per kelas; simpan grafik berdasarkan evaluasi aktual.


In [ ]:
if evaluation is not None:
    matrix = np.asarray(evaluation["confusion_matrix"])
    normalized = np.divide(matrix, matrix.sum(axis=1, keepdims=True), out=np.zeros_like(matrix, dtype=float), where=matrix.sum(axis=1, keepdims=True) != 0)
    names = [name for name, index in sorted(CLASS_MAPPING.items(), key=lambda item: item[1])]
    figure, axes = plt.subplots(1, 2, figsize=(14, 6))
    for axis, data, title in zip(axes, (matrix, normalized), ("Jumlah", "Normalisasi per kelas")):
        image = axis.imshow(data, cmap="Blues"); figure.colorbar(image, ax=axis, fraction=0.046)
        axis.set(xticks=range(8), yticks=range(8), xticklabels=names, yticklabels=names, title=title, xlabel="Prediksi", ylabel="Aktual")
        axis.tick_params(axis="x", labelrotation=60)
    figure.tight_layout(); output = PROJECT_ROOT / "outputs/figures/efficientnet_b2_confusion_matrix.png"
    output.parent.mkdir(parents=True, exist_ok=True); figure.savefig(output, dpi=160); display(figure); plt.close(figure)
else:
    print("Jalankan evaluasi validation dahulu.")

## 20. Training Diagnostics

Heuristik saja: bandingkan gap train-validation dan loss terakhir dengan loss minimum; bukan bukti final generalisasi.


In [ ]:
if phase_rows:
    for row in phase_rows:
        history = load_training_history(PROJECT_ROOT / "outputs/metrics" / f"{row['experiment_id']}.json")
        last = history[-1]; best = min(history, key=lambda item: float(item["val_loss"]))
        gap = float(last["train_accuracy"]) - float(last["val_accuracy"])
        print(row["experiment_id"], "train-val accuracy gap:", round(gap, 4), "last/best val loss:", round(float(last["val_loss"]), 4), "/", round(float(best["val_loss"]), 4))
        print("Indikasi:", "potensi overfitting" if gap > 0.1 and float(last["val_loss"]) > float(best["val_loss"]) else "periksa kurva untuk underfitting atau plateau; tidak ada kesimpulan otomatis")
else:
    print("Tidak ada history untuk diagnosis.")

## 21. Competition Test Audit

Audit hanya `data/competition/test/` dan ID pada sample. Kolom kategori sample bukan ground truth dan tidak digunakan.


In [ ]:
competition_audit = audit_competition_test(project_root=PROJECT_ROOT, expected_count=480)
display(competition_audit.to_dict())

## 22. Standalone Inference Setup

Setelah kernel baru, jalankan **cell ini saja**. Ia menemukan root, memeriksa dependensi, config, checkpoint, transform dan 480 file; tidak melakukan prediksi.


In [ ]:
from pathlib import Path
import sys
import importlib
import warnings
import json
import csv
from collections import Counter

def inference_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data/metadata/split_manifest.csv").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan. Jalankan dari root atau notebooks.")

INFERENCE_ROOT = inference_project_root()
if str(INFERENCE_ROOT) not in sys.path: sys.path.insert(0, str(INFERENCE_ROOT))
for dependency in ("torch", "torchvision", "numpy", "PIL", "yaml", "pandas", "sklearn", "tqdm", "albumentations", "cv2"):
    try: importlib.import_module(dependency)
    except Exception as error: raise ImportError(f"Dependensi inference {dependency!r} tidak tersedia/bermasalah: {error}") from error
import torch
import pandas as pd
from IPython.display import display
from src.data.preprocessing import coerce_rgb_image, load_active_class_mapping
from src.models.efficientnet_b2 import (load_experiment_config, load_selected_checkpoint,
    load_efficientnet_b2_checkpoint, build_efficientnet_b2_transforms, validate_preprocessing,
    EfficientNetB2Error)
from src.inference.competition import (competition_paths, audit_competition_test, load_submission_template,
    CompetitionTestDataset, create_competition_dataloader)
from src.inference.predict import predict_torch_classifier
from src.inference.submission import write_submission_csv, validate_submission_csv

INFERENCE_CONFIG = load_experiment_config(INFERENCE_ROOT)
INFERENCE_MAPPING = load_active_class_mapping(INFERENCE_ROOT)
if len(INFERENCE_MAPPING) != 8: raise ValueError("Mapping kelas aktif harus memiliki delapan kelas.")
INFERENCE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if INFERENCE_DEVICE.type == "cpu": warnings.warn("CUDA tidak tersedia, inference memakai CPU.")
_, INFERENCE_TRANSFORM, INFERENCE_PREPROCESSING = build_efficientnet_b2_transforms()
print("Transform:", validate_preprocessing(INFERENCE_TRANSFORM))
INFERENCE_TEST_DIR, INFERENCE_SAMPLE_PATH = competition_paths(INFERENCE_ROOT)
INFERENCE_TEMPLATE = load_submission_template(INFERENCE_SAMPLE_PATH)
INFERENCE_AUDIT = audit_competition_test(project_root=INFERENCE_ROOT, expected_count=480)
INFERENCE_OUTPUT = INFERENCE_ROOT / "Final Dataset for Submission to Kaggle/submission_efficientnet_b2.csv"
INFERENCE_SELECTED = load_selected_checkpoint(INFERENCE_ROOT)
if INFERENCE_SELECTED.preprocessing != INFERENCE_PREPROCESSING or INFERENCE_SELECTED.class_mapping != INFERENCE_MAPPING:
    raise EfficientNetB2Error("Metadata preprocessing atau class mapping checkpoint tidak cocok.")
INFERENCE_LOADED = load_efficientnet_b2_checkpoint(project_root=INFERENCE_ROOT,
    checkpoint_path=INFERENCE_SELECTED.checkpoint_path, device=INFERENCE_DEVICE, expected_class_mapping=INFERENCE_MAPPING)
if INFERENCE_LOADED.model.classifier[-1].out_features != 8: raise ValueError("Classifier bukan delapan kelas.")
print("Root:", INFERENCE_ROOT, "Device:", INFERENCE_DEVICE, "Checkpoint:", INFERENCE_SELECTED)
print("Audit test:", INFERENCE_AUDIT.to_dict())

## 23. Manual Test Inference

Set flag di cell ini ke `True` hanya ketika siap; cell ini berdiri sendiri setelah section 22, memuat ulang checkpoint dan tidak menimpa CSV yang ada.


In [ ]:
RUN_TEST_INFERENCE = True
if RUN_TEST_INFERENCE:
    INFERENCE_LOADED = load_efficientnet_b2_checkpoint(project_root=INFERENCE_ROOT,
        checkpoint_path=INFERENCE_SELECTED.checkpoint_path, device=INFERENCE_DEVICE, expected_class_mapping=INFERENCE_MAPPING)
    INFERENCE_DATASET = CompetitionTestDataset(model_type="efficientnet", project_root=INFERENCE_ROOT,
        expected_count=480, transform=lambda path: INFERENCE_TRANSFORM(coerce_rgb_image(path)))
    INFERENCE_LOADER = create_competition_dataloader(INFERENCE_DATASET, batch_size=8, num_workers=0)
    try:
        predictions = predict_torch_classifier(INFERENCE_LOADED.model, INFERENCE_LOADER,
            index_to_class=INFERENCE_LOADED.index_to_class, device=INFERENCE_DEVICE)
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("CUDA OOM saat inference. Kurangi batch_size menjadi 4 atau 2 dan jalankan ulang secara manual.") from error
    write_submission_csv(INFERENCE_TEMPLATE, predictions, INFERENCE_OUTPUT, allowed_classes=INFERENCE_MAPPING, overwrite=False)
    validate_submission_csv(INFERENCE_OUTPUT, INFERENCE_TEMPLATE, allowed_classes=INFERENCE_MAPPING)
    print("Submission dibuat:", INFERENCE_OUTPUT)
else:
    print("Inference test tidak dijalankan.")

## 24. Submission CSV Verification

Validasi CSV yang sudah dibuat tanpa menjalankan inference ulang. Tetap memerlukan standalone setup section 22.


In [ ]:
if INFERENCE_OUTPUT.is_file():
    validate_submission_csv(INFERENCE_OUTPUT, INFERENCE_TEMPLATE, allowed_classes=INFERENCE_MAPPING)
    submission = pd.read_csv(INFERENCE_OUTPUT, dtype={"image_id": str})
    assert list(submission.columns) == ["image_id", "waste_category"] and len(submission) == 480
    display(submission.head())
    display(submission["waste_category"].value_counts().reindex(list(INFERENCE_MAPPING), fill_value=0).rename("count"))
    print("CSV valid; urutan ID dan leading zero dipertahankan.")
else:
    print("CSV belum ada; inference belum dijalankan.")

## 25. Experiment Summary

Ringkasan artefak dan status; tidak mengklaim model telah dilatih atau CSV telah dibuat.


In [ ]:
print("Dataset lama V1: eksperimen teknis, bukan kepatuhan aturan lima label manual per kelas.")
print("Checkpoints:", PROJECT_ROOT / "models/checkpoints" / HEAD_EXPERIMENT_ID, "dan", PROJECT_ROOT / "models/checkpoints" / FINETUNE_EXPERIMENT_ID)
print("History:", PROJECT_ROOT / "outputs/metrics", "| Log:", PROJECT_ROOT / "outputs/logs", "| Grafik:", PROJECT_ROOT / "outputs/figures")
print("Submission target:", PROJECT_ROOT / "Final Dataset for Submission to Kaggle/submission_efficientnet_b2.csv")